# 🐖 20 | Interpretabilidad de segmentación por instancias: Grad-CAM orientado a máscaras para **el mismo animal**

**RECURSO: GPU T4** — notebook independiente. **No entrena modelos.** Utiliza exclusivamente imágenes de **validación** y los mejores pesos congelados del proyecto. Una imagen por fecha, seleccionada de forma determinista ANTES de ver las activaciones.

El objetivo no es generar mapas de clasificación genéricos, sino mapas **asociados a un candidato de máscara** de YOLO11n-seg y a una región de interés de Mask R-CNN.

* YOLO: Grad-CAM del bloque compartido `model.model[4]` usando como objetivo diferenciable la confianza del candidato **emparejado por caja** multiplicada por la probabilidad media de su máscara en la región correspondiente. Si no se consigue ese emparejamiento, NO se fabrica un mapa.
* Mask R-CNN: ROI Grad-CAM del cabezal de máscaras usando la probabilidad media de máscara de la detección elegida. Se proyecta al recuadro de esa instancia. Esta **ROI-CAM no debe compararse píxel a píxel** con la CAM de campo completo de YOLO.

**No son explicaciones causales, no prueban saliencia humana ni oclusión real.** Los ejemplos cualitativos no respaldan una afirmación de significación estadística. Verificar licencia CC BY-NC 4.0 de FaroPigSeg antes de reproducir imágenes en una revista.
**Corrección de ejecución (18-10-2026):** se habilita `requires_grad` en la entrada de YOLO para calcular Grad-CAM con pesos congelados. Los pesos, la selección de muestras, el protocolo de evaluación y los resultados anteriores no se modifican. La versión anterior se archiva para evitar ejecutar la celda defectuosa.

**Corrección v3 (20-10-2026):** lectura compatible con variantes de salida de Ultralytics mediante captura directa de prototipos; preflight de una imagen; diagnóstico exportable si falla. Mantiene pesos originales y solo usa validación.

**Revisión metodológica V4:** para cada imagen se elige una instancia de referencia anotada en el GT de validación; ambas atribuciones deben corresponder a esa misma instancia. Se registran los identificadores, IoU y umbrales, sin reentrenar modelos.

**Sesgo de selección:** las figuras ilustran instancias localizadas correctamente por los dos modelos, elegidas antes de calcular Grad-CAM; no representan casos de fallo ni permiten medir la calidad global del modelo.


## A. GPU, paquetes y rutas

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Notebook 20 requiere GPU: Entorno de ejecución > Cambiar tipo > GPU T4')
print('✅ GPU activa:',torch.cuda.get_device_name(0))


In [ ]:
%pip -q install "ultralytics>=8.3,<9" "pyyaml>=6,<7" "openpyxl>=3.1" "pandas>=2,<3" "opencv-python-headless>=4.7" matplotlib pillow "pycocotools>=2.0.8"


In [ ]:
from google.colab import drive,files
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import json,hashlib,zipfile,sys,traceback,platform,gc
import numpy as np,pandas as pd,cv2,yaml,torch,torchvision
from PIL import Image
import matplotlib.pyplot as plt
from pycocotools import mask as maskUtils
from ultralytics import YOLO
from ultralytics.data.augment import LetterBox
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.transforms.functional import to_tensor

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
DATA=ROOT/'01_Datasets'/'FaroPigSeg'; REPORTS=ROOT/'03_Resultados'/'05_Reportes'
FIG=ROOT/'03_Resultados'/'04_Figuras';MET=ROOT/'03_Resultados'/'03_Metricas';EXP=ROOT/'05_Exportaciones_ZIP'
RUN=ROOT/'03_Resultados'/'06_Entrenamiento'
YAML=DATA/'split_fecha_v1'/'data_fecha.yaml'
YOLO_W=RUN/'YOLO11n_fecha_v1'/'weights'/'best.pt'
MASK_W=RUN/'MaskRCNN_fecha_v1'/'maskrcnn_best_model.pt'
PROTO09=REPORTS/'CERDOS_09_COCO_Armonizado'/'CERDOS_09_Protocolo.json'
for p in [YAML,YOLO_W,MASK_W,PROTO09]:
    if not p.is_file():raise FileNotFoundError('Falta archivo necesario: '+str(p))
for p in [FIG,MET,REPORTS,EXP]:p.mkdir(parents=True,exist_ok=True)

def sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
old=json.loads(PROTO09.read_text(encoding='utf8'))
assert sha256(YOLO_W)==old['yolo_sha256']
assert sha256(MASK_W)==old['mask_sha256']
assert sha256(YAML)==old['yaml_sha256']
config=yaml.safe_load(YAML.read_text(encoding='utf8'))
val_file=Path(config['val']);assert val_file.is_file()
paths=[Path(x.strip()) for x in val_file.read_text(encoding='utf8').splitlines() if x.strip()]
assert len(paths)==196 and all(p.is_file() for p in paths)
by_date=defaultdict(list)
for p in paths:by_date[p.name[:10]].append(p)
assert len(by_date)==4
SAMPLES=[sorted(by_date[d])[len(by_date[d])//2] for d in sorted(by_date)]
assert len(SAMPLES)==4
PROTO={'pipeline':'CERDOS_20_same_gt_instance_v4',
 'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__,
 'torchvision':torchvision.__version__,'ultralytics':__import__('ultralytics').__version__,
 'yaml_sha256':old['yaml_sha256'],'yolo_sha256':old['yolo_sha256'],'mask_sha256':old['mask_sha256'],
 'sampling':'middle sorted validation filename per capture date; choose GT object with highest minimum YOLO and Mask R-CNN mask IoU in cached val predictions',
 'files':[p.name for p in SAMPLES], 'yolo_layer':'model.model[4]',
 'yolo_target':'mask score for the detection matching the SAME GT reference selected for both models',
 'yolo_match_box_IoU_min':0.35, 'yolo_size':640,'yolo_conf_display':0.25,
 'mask_target':'mean sigmoid mask logit, class pig, ROI matched to same GT as YOLO',
 'mask_layer':'roi_heads.mask_head','mask_min_size':512,'mask_max_size':768,
 'gt_match_min_iou':0.50,'notes':['Grad-CAM is an exploratory local attribution, not a causal explanation',
          'Mask R-CNN ROI heatmap spatial support differs from YOLO full-frame CAM',
          'No unverified source/camera/domain claims','Only validation images; no test used']}
JPROTO=REPORTS/'CERDOS_20_Misma_Instancia_Protocolo_V4.json'
# Artefactos de validación del notebook 13: contienen GT y las predicciones sin reentrenamiento.
SAVE13=REPORTS/'CERDOS_13_Calibracion_Validacion'
GT13=SAVE13/'CERDOS_13_GT_val_coco.json'
Y13=SAVE13/'CERDOS_13_Pred_YOLO_val_coco.json'
M13=SAVE13/'CERDOS_13_Pred_MaskRCNN_val_coco.json'
for file in (GT13,Y13,M13):
    if not file.is_file():raise FileNotFoundError('Falta archivo de predicciones/GT de validación 13: '+str(file))
GT_DATA=json.loads(GT13.read_text(encoding='utf8'))
Y_DATA=json.loads(Y13.read_text(encoding='utf8'))
M_DATA=json.loads(M13.read_text(encoding='utf8'))
BY_NAME={im['file_name']:im for im in GT_DATA['images']}
BY_GT=defaultdict(list);BY_Y=defaultdict(list);BY_M=defaultdict(list)
for g in GT_DATA['annotations']:BY_GT[g['image_id']].append(g)
for p in Y_DATA:
    if float(p['score'])>=0.25:BY_Y[p['image_id']].append(p)
for p in M_DATA:
    if float(p['score'])>=0.25:BY_M[p['image_id']].append(p)
PROTO['gt13_sha256']=sha256(GT13)
PROTO['y13_sha256']=sha256(Y13)
PROTO['m13_sha256']=sha256(M13)
# Comprobar el protocolo FINAL con hashes completos antes de escribir.
if JPROTO.is_file() and json.loads(JPROTO.read_text(encoding='utf8'))!=PROTO:
    raise RuntimeError('Ya existe un protocolo V4 diferente en Drive: no sobrescribir')
JPROTO.write_text(json.dumps(PROTO,ensure_ascii=False,indent=2),encoding='utf8')
print('✅ Val:',len(paths),'imágenes. Cuatro casos prefijados:',[p.name for p in SAMPLES])


## B. Funciones matemáticas y visualización reproducible

In [ ]:
def norm_cam(x):
    a=np.asarray(x,dtype=np.float32)
    a=np.nan_to_num(a,nan=0.,posinf=0.,neginf=0.)
    a=np.maximum(a,0.)
    hi=float(np.percentile(a,99.5))
    if hi<=1.e-10:raise ValueError('Gradientes nulos / CAM no informativa; NO presentar mapa.')
    return np.clip(a/hi,0,1)

def box_iou_against_one(boxes,box):
    xx1=torch.maximum(boxes[:,0],box[0]); yy1=torch.maximum(boxes[:,1],box[1])
    xx2=torch.minimum(boxes[:,2],box[2]); yy2=torch.minimum(boxes[:,3],box[3])
    inter=(xx2-xx1).clamp(min=0)*(yy2-yy1).clamp(min=0)
    a=(boxes[:,2]-boxes[:,0]).clamp(min=0)*(boxes[:,3]-boxes[:,1]).clamp(min=0)
    b=(box[2]-box[0]).clamp(min=0)*(box[3]-box[1]).clamp(min=0)
    return inter/(a+b-inter+1e-7)

def masked_display(rgb,mask):
    out=rgb.astype(np.float32).copy()
    pix=np.asarray(mask).astype(bool)
    if pix.shape!=rgb.shape[:2]:pix=cv2.resize(pix.astype('uint8'),(rgb.shape[1],rgb.shape[0]),interpolation=cv2.INTER_NEAREST)>0
    color=np.array([35,185,141],dtype=np.float32)
    out[pix]=out[pix]*.60+color*.40
    return np.uint8(np.clip(out,0,255))

def salience_display(rgb,heat):
    h=norm_cam(heat)
    cmap=cv2.applyColorMap(np.uint8(h*255),cv2.COLORMAP_TURBO)[:,:,::-1].astype(np.float32)
    out=rgb.astype(np.float32).copy()
    loc=h>.02
    weight=(h*.55).astype(np.float32)[:,:,None]
    blend=out*(1-weight)+cmap*weight
    out[loc]=blend[loc]
    return np.uint8(np.clip(out,0,255))

def load_rgb(p):
    with Image.open(p) as im:return np.asarray(im.convert('RGB')).copy()

plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.titlesize':11,
                     'svg.fonttype':'none','savefig.facecolor':'white'})


def pixel_iou(mask_a,mask_b):
    a=np.asarray(mask_a,dtype=bool);b=np.asarray(mask_b,dtype=bool)
    if a.shape!=b.shape:raise ValueError(f'Máscaras con distintas dimensiones: {a.shape} != {b.shape}')
    intersect=int(np.count_nonzero(a&b));union=int(np.count_nonzero(a|b))
    return float(intersect/union) if union else 0.0

def choose_gt_reference(path):
    """Escoge un GT válido para AMBOS modelos, según predicciones de validación guardadas."""
    im=BY_NAME.get(path.name)
    if im is None:raise ValueError('Imagen ausente del GT de validación 13: '+path.name)
    imid=int(im['id']);gts=BY_GT[imid]
    ys=BY_Y[imid];ms=BY_M[imid]
    if not ys or not ms:raise ValueError('Faltan detecciones en alguno de los modelos')
    # Decodificar RLE COCO de la fase 13 (sin recalcular anotaciones).
    y_masks=[maskUtils.decode(p['segmentation']).astype(bool) for p in ys]
    m_masks=[maskUtils.decode(p['segmentation']).astype(bool) for p in ms]
    matches=[]
    for gt in gts:
        gmask=maskUtils.decode(gt['segmentation']).astype(bool)
        yi=[pixel_iou(gmask,z) for z in y_masks]
        mi=[pixel_iou(gmask,z) for z in m_masks]
        yi_max=max(yi);mi_max=max(mi)
        if min(yi_max,mi_max)<PROTO['gt_match_min_iou']:continue
        matches.append((min(yi_max,mi_max), (yi_max+mi_max)/2, -int(gt['id']), gt,gmask,yi_max,mi_max))
    if not matches:
        raise ValueError('No existe un cerdo GT detectado por ambos modelos con IoU >= 0.50 en validación')
    chosen=max(matches,key=lambda z:z[:3]);g=chosen[3]
    return {'gt_id':int(g['id']),'image_id':imid,'gt_mask':chosen[4],
            'val_yolo_mask_iou':float(chosen[5]),'val_maskrcnn_mask_iou':float(chosen[6]),
            'candidate_gt_count':len(matches)}


## C. YOLO11n-seg: Grad-CAM con objetivo de máscara específico

In [ ]:
YM=YOLO(str(YOLO_W))
YM.model.to('cuda:0').eval()

# Preprocesado fijo 640x640, sin modo rectangular. Se replica para inferencia y gradientes.
def yolo_roi_cam(rgb,gt_mask):
    h,w=rgb.shape[:2]
    bgr=cv2.cvtColor(rgb,cv2.COLOR_RGB2BGR)
    with torch.no_grad():
        result=YM.predict(bgr,device=0,imgsz=640,rect=False,
              conf=0.25,iou=float(old['yolo_iou_nms']),max_det=int(old['max_dets']),
              retina_masks=True,half=False,save=False,verbose=False)[0]
    if result.boxes is None or len(result.boxes)<1 or result.masks is None:
        raise ValueError('YOLO sin máscara seleccionable a confianza >= 0.25')
    pred_masks=[(z.detach().cpu().numpy()>0.5) for z in result.masks.data]
    iou_to_gt=[pixel_iou(z,gt_mask) for z in pred_masks]
    best=int(np.argmax(iou_to_gt))
    if iou_to_gt[best]<PROTO['gt_match_min_iou']:
        raise ValueError(f'YOLO no coincide con el GT común: mejor IoU={iou_to_gt[best]:.3f}')
    boxref=result.boxes.xyxy[best].to('cuda').float()
    maskpred=pred_masks[best]

    input_bgr=LetterBox(new_shape=(640,640),auto=False,stride=32)(image=bgr)
    assert input_bgr.shape[:2]==(640,640)
    x=torch.from_numpy(cv2.cvtColor(input_bgr,cv2.COLOR_BGR2RGB).transpose(2,0,1).copy()).unsqueeze(0).float().to('cuda')/255.
    # Los pesos de YOLO cargados para inferencia pueden estar congelados.
    # Grad-CAM necesita gradientes respecto a la entrada para que las
    # activaciones de backbone tambien conserven el grafo de autograd.
    # NO entrena, modifica ni actualiza los pesos del modelo.
    x=x.detach().requires_grad_(True)
    act=[]
    prototypes=[]
    # No deducir el tensor de prototipos de índices de una tupla: el contrato
    # de salida de Ultralytics cambia entre versiones. Se captura en su módulo.
    head=YM.model.model[-1]
    nm=int(getattr(head,'nm',0));nc=int(getattr(head,'nc',0))
    if nm<=0 or nc!=1 or not hasattr(head,'proto'):
        raise RuntimeError(f'Cabezal YOLO incompatible: {type(head).__name__}, nm={nm}, nc={nc}')
    def save_proto(module,inputs,output):
        prototypes.append(output)
    def save_act(m,i,o):
        if not torch.is_tensor(o):raise RuntimeError('Capa YOLO no devuelve Tensor')
        if not o.requires_grad:raise RuntimeError('Activación sin gradientes: confirmar enable_grad y entrada con gradiente')
        o.retain_grad();act.append(o)
    hook=YM.model.model[4].register_forward_hook(save_act)
    hookproto=head.proto.register_forward_hook(save_proto)
    def leaf_tensors(value,depth=0):
        if depth>8:return
        if torch.is_tensor(value):
            yield value
        elif isinstance(value,(tuple,list)):
            for item in value:yield from leaf_tensors(item,depth+1)
        elif isinstance(value,dict):
            for item in value.values():yield from leaf_tensors(item,depth+1)
    try:
        with torch.enable_grad():
            raw=YM.model(x)
            if not prototypes:
                raise RuntimeError('No se ejecutó el módulo YOLO proto; revisar cabezal de segmentación')
            protos=[t for t in leaf_tensors(prototypes[-1])
                    if t.ndim==4 and t.shape[0]==1 and t.shape[1]==nm]
            if not protos:
                raise RuntimeError('La salida del módulo proto no tiene forma [1,nm,H,W]')
            proto=protos[0]
            if not proto.requires_grad:
                raise RuntimeError('Prototipos sin grafo autograd; revisar estado de inferencia del modelo')
            channels=4+nc+nm
            preds=[]
            for tensor in leaf_tensors(raw):
                if tensor.ndim!=3 or tensor.shape[0]!=1:continue
                if tensor.shape[1]==channels and tensor.shape[2]>100:
                    preds.append(tensor)
                elif tensor.shape[2]==channels and tensor.shape[1]>100:
                    preds.append(tensor.transpose(1,2))
            if not preds:
                # Diagnóstico real de formas en el error, útil si la API cambia.
                summary=[tuple(t.shape) for t in leaf_tensors(raw)]
                raise RuntimeError(f'Sin tensor de predicción [1,{channels},N] en YOLO. Formas={summary[:20]}')
            pred=preds[0]
            if not pred.requires_grad:
                raise RuntimeError('Predicciones sin grafo autograd (modelo en inference_mode)')
            # Tanto Segment.forward moderno como clásico contienen los canales:
            # xywh (4), confianza por clase (nc) y coeficientes de máscara (nm).
            nmask=int(proto.shape[1]);nclass=int(pred.shape[1]-4-nmask)
            if nclass!=1:raise RuntimeError('Se esperaba una clase cerdo en YOLO; clases='+str(nclass))
            boxes=pred[0,:4,:].T.float()
            # Datos decodificados YOLO: centros xywh en imagen letterboxed 640.
            xyxy=torch.stack([boxes[:,0]-boxes[:,2]/2,boxes[:,1]-boxes[:,3]/2,
                              boxes[:,0]+boxes[:,2]/2,boxes[:,1]+boxes[:,3]/2],dim=1)
            r=min(640/h,640/w);nh=int(round(h*r));nw=int(round(w*r))
            top=int(round((640-nh)/2-.1));left=int(round((640-nw)/2-.1))
            ref=torch.stack([boxref[0]*r+left,boxref[1]*r+top,
                             boxref[2]*r+left,boxref[3]*r+top])
            ious=box_iou_against_one(xyxy,ref)
            scores=pred[0,4,:]
            candidates=torch.where(ious>=.35,scores,torch.full_like(scores,-1.))
            idx=int(candidates.argmax().item());matched=float(ious[idx].detach().cpu())
            if matched<.35:raise ValueError('No coincide candidato pre-NMS con instancia post-NMS')
            coef=pred[0,5:5+nmask,idx]
            masklogits=(coef[:,None,None]*proto[0]).sum(dim=0)
            ph,pw=masklogits.shape
            bx=xyxy[idx].detach()
            x1=max(0,min(pw-1,int(torch.floor(bx[0]*pw/640).item())))
            x2=max(x1+1,min(pw,int(torch.ceil(bx[2]*pw/640).item())))
            y1=max(0,min(ph-1,int(torch.floor(bx[1]*ph/640).item())))
            y2=max(y1+1,min(ph,int(torch.ceil(bx[3]*ph/640).item())))
            target=scores[idx]*torch.sigmoid(masklogits[y1:y2,x1:x2]).mean()
            YM.model.zero_grad(set_to_none=True)
            target.backward()
            if not act or act[0].grad is None:raise RuntimeError('Sin gradiente de la característica YOLO seleccionada')
            a=act[0].detach()[0].float();g=act[0].grad.detach()[0].float()
            heat=torch.relu((g.mean(dim=(1,2),keepdim=True)*a).sum(dim=0)).cpu().numpy()
            heat=cv2.resize(heat,(640,640),interpolation=cv2.INTER_LINEAR)
            heat=cv2.resize(heat[top:top+nh,left:left+nw],(w,h),interpolation=cv2.INTER_LINEAR)
            heat=norm_cam(heat)
            info={'selected_confidence':float(result.boxes.conf[best].item()),
                  'iou_with_same_gt':float(iou_to_gt[best]),'detection_index':best,
                  'raw_candidate_iou_with_output_box':matched,
                  'grad_target':float(target.detach().item()),'selected_bbox_xyxy':boxref.detach().cpu().tolist(),
                  'attribution_region':'whole image (backbone feature model[4])'}
            return maskpred,heat,info
    finally:
        hook.remove();hookproto.remove();YM.model.zero_grad(set_to_none=True)


## D. Mask R-CNN: gradientes de máscara sobre ROI

In [ ]:
def new_maskrcnn():
    m=maskrcnn_resnet50_fpn(weights=None,weights_backbone=None,
       min_size=512,max_size=768,trainable_backbone_layers=3)
    nf=m.roi_heads.box_predictor.cls_score.in_features
    m.roi_heads.box_predictor=FastRCNNPredictor(nf,2)
    nfeat=m.roi_heads.mask_predictor.conv5_mask.in_channels
    m.roi_heads.mask_predictor=MaskRCNNPredictor(nfeat,256,2)
    m.roi_heads.score_thresh=.25
    m.roi_heads.detections_per_img=int(old['max_dets'])
    return m

MM=new_maskrcnn().to('cuda:0')
state=torch.load(MASK_W,map_location='cpu',weights_only=False)
MM.load_state_dict(state['model'],strict=True)
del state
MM.eval()

def mask_roi_gradcam(rgb,gt_mask):
    h,w=rgb.shape[:2]
    acts=[];logits=[]
    def hook_a(m,i,o):
        o.retain_grad();acts.append(o)
    def hook_l(m,i,o):logits.append(o)
    h1=MM.roi_heads.mask_head.register_forward_hook(hook_a)
    h2=MM.roi_heads.mask_predictor.register_forward_hook(hook_l)
    try:
        x=to_tensor(rgb).to('cuda:0')
        with torch.enable_grad():
            out=MM([x])[0]
            if len(out['scores'])==0:raise ValueError('Mask R-CNN sin detecciones')
            det_masks=(out['masks'][:,0].detach().cpu().numpy()>.5)
            iou_to_gt=[pixel_iou(z,gt_mask) for z in det_masks]
            idx=int(np.argmax(iou_to_gt))
            if iou_to_gt[idx]<PROTO['gt_match_min_iou']:
                raise ValueError(f'Mask R-CNN no coincide con el GT común: IoU={iou_to_gt[idx]:.3f}')
            if not acts or not logits or logits[0].shape[0]!=len(out['scores']):
                raise RuntimeError('Índices ROI de máscara diferentes de predicciones; no atribuir')
            a=acts[0]; logit=logits[0]
            if logit.shape[1]<2:raise RuntimeError('Se requiere canal de clase cerdo')
            target=torch.sigmoid(logit[idx,1,:,:]).mean()
            MM.zero_grad(set_to_none=True);target.backward()
            if a.grad is None:raise RuntimeError('Sin gradiente ROI')
            feat=a.detach()[idx].float();grad=a.grad.detach()[idx].float()
            cam=torch.relu((grad.mean(dim=(1,2),keepdim=True)*feat).sum(dim=0)).cpu().numpy()
            roi=norm_cam(cam)
            box=out['boxes'][idx].detach().cpu().numpy()
            x1=max(0,min(w-1,int(np.floor(box[0]))));x2=max(x1+1,min(w,int(np.ceil(box[2]))))
            y1=max(0,min(h-1,int(np.floor(box[1]))));y2=max(y1+1,min(h,int(np.ceil(box[3]))))
            heat=np.zeros((h,w),dtype=np.float32)
            heat[y1:y2,x1:x2]=cv2.resize(roi,(x2-x1,y2-y1),interpolation=cv2.INTER_LINEAR)
            mask=(out['masks'][idx,0].detach().cpu().numpy()>.5)
            info={'selected_confidence':float(out['scores'][idx].detach().cpu()),
                  'iou_with_same_gt':float(iou_to_gt[idx]),'detection_index':idx,
                  'grad_target':float(target.detach().cpu()),
                  'selected_bbox_xyxy':[float(v) for v in box],
                  'attribution_region':'Mask R-CNN selected ROI; outside bounding box is NOT evaluated'}
            return mask,heat,info
    finally:
        h1.remove();h2.remove();MM.zero_grad(set_to_none=True)


## E. Generar las figuras, conservar fallos y exportar

In [ ]:
# Preflight: una sola imagen. Evita repetir cuatro fallos consecutivos.
# Si esta prueba falla, exporta un diagnóstico sin alterar los modelos.
DIAG_PATH=REPORTS/'CERDOS_20_Diagnostico_V4.json'
PREFLIGHT_OK=False
try:
    pre_rgb=load_rgb(SAMPLES[0])
    pre_ref=choose_gt_reference(SAMPLES[0])
    test_ym,test_yc,test_yi=yolo_roi_cam(pre_rgb,pre_ref['gt_mask'])
    test_mm,test_mc,test_mi=mask_roi_gradcam(pre_rgb,pre_ref['gt_mask'])
    if not np.isfinite(test_yc).all() or not np.isfinite(test_mc).all():
        raise ValueError('CAM preflight no finita')
    PREFLIGHT_OK=True
    print('✅ Preflight: ambos modelos explican GT',pre_ref['gt_id'],'en',SAMPLES[0].name)
    print('YOLO: confianza=',test_yi['selected_confidence'],'IoU candidato=',test_yi['raw_candidate_iou_with_output_box'])
    print('Mask R-CNN: confianza=',test_mi['selected_confidence'])
except Exception as exc:
    diag={'etapa':'preflight','imagen':SAMPLES[0].name,
          'error':repr(exc),'traceback':traceback.format_exc(),
          'torch':torch.__version__,'ultralytics':__import__('ultralytics').__version__,
          'head':type(YM.model.model[-1]).__name__,
          'nc':int(getattr(YM.model.model[-1],'nc',-1)),
          'nm':int(getattr(YM.model.model[-1],'nm',-1)),
          'gpu':torch.cuda.get_device_name(0)}
    DIAG_PATH.write_text(json.dumps(diag,indent=2,ensure_ascii=False),encoding='utf-8')
    DIAG_ZIP=EXP/'CERDOS_20_Diagnostico_Si_Falla.zip'
    with zipfile.ZipFile(DIAG_ZIP,'w',zipfile.ZIP_DEFLATED) as z:z.write(DIAG_PATH,arcname=DIAG_PATH.name)
    print('❌ Preflight falló, guardé diagnóstico en:',DIAG_ZIP)
    print('Envía este archivo para corregir el fallo concreto. No se generaron CAM inventados.')
    try:files.download(str(DIAG_ZIP))
    except Exception:pass
    raise


In [ ]:
FIG_DIR=FIG/'CERDOS_20_GradCAM_Misma_Instancia'
FIG_DIR.mkdir(parents=True,exist_ok=True)
RECORDS=[];OUTPUTS=[]
for p in SAMPLES:
    rgb=load_rgb(p)
    record={'imagen':p.name,'fecha':p.name[:10],'familia':'all_in_one' if 'all_in_one' in p.name else ('left_cropped' if 'left_cropped' in p.name else 'sin_clasificar')}
    ref=pre_ref if p==SAMPLES[0] and PREFLIGHT_OK else choose_gt_reference(p)
    record.update({key:val for key,val in ref.items() if key!='gt_mask'})
    try:
        ym,yc,yi=(test_ym,test_yc,test_yi) if p==SAMPLES[0] and PREFLIGHT_OK else yolo_roi_cam(rgb,ref['gt_mask'])
        record.update({'YOLO_estado':'ok',**{'YOLO_'+k:v for k,v in yi.items()}})
    except Exception as ex:
        ym=yc=None;record.update({'YOLO_estado':'fallo','YOLO_error':f'{type(ex).__name__}: {ex}'})
        print('⚠️ YOLO',p.name,record['YOLO_error'])
    try:
        mm,mc,mi=(test_mm,test_mc,test_mi) if p==SAMPLES[0] and PREFLIGHT_OK else mask_roi_gradcam(rgb,ref['gt_mask'])
        record.update({'MaskRCNN_estado':'ok',**{'MaskRCNN_'+k:v for k,v in mi.items()}})
    except Exception as ex:
        mm=mc=None;record.update({'MaskRCNN_estado':'fallo','MaskRCNN_error':f'{type(ex).__name__}: {ex}'})
        print('⚠️ MaskRCNN',p.name,record['MaskRCNN_error'])
    RECORDS.append(record)
    if yc is None or mc is None:continue  # Nunca fabricar una figura CAM en caso de fallo.
    fig,axs=plt.subplots(2,4,figsize=(15.0,6.2),sharex=True,sharey=True)
    pics=[rgb,masked_display(rgb,ref['gt_mask']),masked_display(rgb,ym),salience_display(rgb,yc),
          rgb,masked_display(rgb,ref['gt_mask']),masked_display(rgb,mm),salience_display(rgb,mc)]
    titles=['Input image','Same reference pig (GT)','YOLO | matched mask','YOLO | mask-aware Grad-CAM',
            'Input image','Same reference pig (GT)','Mask R-CNN | matched mask','Mask R-CNN | ROI Grad-CAM']
    for ax,pic,title in zip(axs.flat,pics,titles):ax.imshow(pic);ax.set_title(title);ax.axis('off')
    fig.suptitle(f'Same pig (GT {ref["gt_id"]}) | validation date {p.name[:10]}',fontsize=14,fontweight='semibold')
    fig.subplots_adjust(left=.01,right=.99,bottom=.01,top=.89,wspace=.04,hspace=.16)
    stem='CERDOS_20_Misma_Instancia_'+p.stem
    for ext in ['png','pdf','svg']:
        dst=FIG_DIR/(stem+'.'+ext)
        fig.savefig(dst,dpi=600 if ext=='png' else None,bbox_inches='tight')
        OUTPUTS.append(dst)
    plt.show();plt.close(fig)
    print('✅ Misma instancia',ref['gt_id'],'| YOLO IoU:',round(yi['iou_with_same_gt'],3),'| Mask IoU:',round(mi['iou_with_same_gt'],3))

REPS=pd.DataFrame(RECORDS)
EXCEL=MET/'CERDOS_20_Interpretabilidad_Misma_Instancia.xlsx'
with pd.ExcelWriter(EXCEL,engine='openpyxl') as w:
    REPS.to_excel(w,sheet_name='Casos_Validacion',index=False)
    pd.DataFrame([PROTO]).to_excel(w,sheet_name='Protocolo',index=False)
    pd.DataFrame({'advertencia':PROTO['notes']}).to_excel(w,sheet_name='Limitaciones',index=False)
if not OUTPUTS:
    raise RuntimeError('Ninguna imagen pudo producir ambos Grad-CAM válidos. Revisar Excel y mensajes; NO publicar mapas fabricados.')
ZIP=EXP/'CERDOS_20_GradCAM_Misma_Instancia_Resultados.zip'
with zipfile.ZipFile(ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in [EXCEL,JPROTO]+OUTPUTS:z.write(p,arcname=p.name)
with zipfile.ZipFile(ZIP) as z:assert z.testzip() is None
print('✅ Figuras válidas:',len(OUTPUTS)//3,'de',len(SAMPLES),'| ZIP:',ZIP)
files.download(str(ZIP))


**Importante:** interpretar únicamente los mapas calculados sin error. Para la publicación, atribuir FaroPigSeg y verificar la licencia de las imágenes. Las CAM no prueban qué píxeles causaron la detección.